In [1]:
# Importy
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import joblib

C:\Users\chlip\anaconda3\envs\bim-nlp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [42]:
# Wczytanie danych z etykietami
df = pd.read_csv("../data/extra_layer.csv")
print("Liczba przykładów:", len(df))
df = df.dropna(subset=["Text", "ClassificationCode"])

texts = df["Text"].tolist()
labels = df["ClassificationCode"].tolist()

Liczba przykładów: 11
                  GlobalId              IfcType  \
0   1iXqOKn4z17Aflw5Qn1m$o              IFCWALL   
1   0ref1xZSbESRxnXlPr5A_Q            IFCCOLUMN   
2   0zM8mWnon67BszuE8pp38x       IFCCURTAINWALL   
3   3vo48kV4b1MQvxT$C2APPu             IFCSTAIR   
4   39aoUwyvH3aBqT6fn2tKna              IFCSLAB   
5   01rQZTvw15RvGY16dC6fBl              IFCBEAM   
6   06njhvWIbET8dSE8y$eyqB           IFCRAILING   
7   2PCnyP6NPEBhDTr6LLyNlc  IFCWALLSTANDARDCASE   
8   2PCnyP6NPEBhDTr6LLyNlc  IFCWALLSTANDARDCASE   
9   1IDH6qz$5CXRCQhX08JFW5              IFCBEAM   
10  1_AeLI3Ej5seVggccHPkMZ              IFCDOOR   

                                                 Name  \
0                                                 SZ1   
1                                                  S1   
2                                                 SK1   
3                                                SCH1   
4                                                 DA1   
5                      

In [44]:
# Enkoder etykiet
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)

In [46]:
# Podział na zbiory treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(texts, y, test_size=0.2, random_state=42, stratify=y)

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [24]:
# Wczytanie modelu po TSDAE i few-shot fine-tuning
model = SentenceTransformer("../models/fewshot_finetuned")

In [25]:
# Embeddingi
X_train_emb = model.encode(X_train, batch_size=16, convert_to_numpy=True)
X_test_emb = model.encode(X_test, batch_size=16, convert_to_numpy=True)

C:\Users\chlip\anaconda3\envs\bim-nlp\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [26]:
# Trening klasyfikatora (tu: regresja logistyczna)
clf = LogisticRegression(max_iter=1000, class_weight="balanced")
clf.fit(X_train_emb, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [16]:
# Ewaluacja
y_pred = clf.predict(X_test_emb)
print("Raport klasyfikacji:\n")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

print("Macierz pomyłek:\n")
print(confusion_matrix(y_test, y_pred))

Raport klasyfikacji:

              precision    recall  f1-score   support

    EF_25_10       1.00      1.00      1.00        59
    EF_25_55       1.00      1.00      1.00         1
       EF_30       1.00      1.00      1.00         1
    EF_30_20       1.00      1.00      1.00         2
 EF_35_10_40       1.00      1.00      1.00         2
    Ss_20_30       1.00      1.00      1.00        10
    Ss_25_12       1.00      1.00      1.00        24

    accuracy                           1.00        99
   macro avg       1.00      1.00      1.00        99
weighted avg       1.00      1.00      1.00        99

Macierz pomyłek:

[[59  0  0  0  0  0  0]
 [ 0  1  0  0  0  0  0]
 [ 0  0  1  0  0  0  0]
 [ 0  0  0  2  0  0  0]
 [ 0  0  0  0  2  0  0]
 [ 0  0  0  0  0 10  0]
 [ 0  0  0  0  0  0 24]]


In [17]:
# Zapis modelu z warstwą klasyfikacyjną
joblib.dump(clf, "../models/classifier_logreg.joblib")
joblib.dump(label_encoder, "../models/label_encoder.joblib")

print("Zapisano klasyfikator i enkoder etykiet.")

Zapisano klasyfikator i enkoder etykiet.


In [19]:
# Test
from sentence_transformers import SentenceTransformer
import joblib

model = SentenceTransformer("../models/fewshot_finetuned")
clf = joblib.load("../models/classifier_logreg.joblib")
label_encoder = joblib.load("../models/label_encoder.joblib")

# Nowe dane (bez etykiet)
new_df = pd.read_csv("../data/ifc_objects.csv")
texts = new_df["Text"].tolist()

# Embeddingi i predykcja
embeddings = model.encode(texts, convert_to_numpy=True)
preds = clf.predict(embeddings)
pred_labels = label_encoder.inverse_transform(preds)

# Zapisz wyniki
new_df["PredictedLabel"] = pred_labels
new_df.to_csv("../results/hybrid_test.csv", index=False)

C:\Users\chlip\anaconda3\envs\bim-nlp\lib\site-packages\torch\nn\modules\module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [20]:
display(new_df)

,GlobalId,IfcType,Name,Text,PredictedLabel
0,3zR0BOEcLADRKln4HYporH,IFCSLAB,floor,"A solid, site-cast concrete floor, providing a...",EF_25_55
1,1AQAupaRP1txwK1AGiN61V,IFCWALL,house - outer wall - house right front,"A solid outer wall, forming the right front si...",EF_25_55
2,3wdauVJT5Fx9drrREiDqA$,IFCWALL,house - outer wall - house right back,"A solid outer wall, forming the right back sid...",EF_30_20
3,0OfZwWc8j9QP5uX8xPTxDH,IFCWALL,house - outer wall - house left,"A solid outer wall, forming the left side of t...",EF_25_55
4,1uS5vfZPn9R8PlAaVd73on,IFCWALL,plumbing wall,A wall designed to house and protect plumbing ...,EF_30_20
5,0ZTBBPo6f6bxqV2K7Oelrq,IFCSLAB,house - roof - slab left,A roof slab that's got it all covered. IsExter...,EF_25_55
6,12UVOn4wvAJPMUExKdZLb8,IFCSLAB,house - roof - slab right,A roof slab that's got it all covered. IsExter...,EF_25_55
